# Arabic Classical ML Preprocessing — WITH Stemming

**Pipeline:**
1. Install & import dependencies
2. Configuration block
3. Regex / constant definitions
4. Helper preprocessing functions
5. Stemming function (ISRIStemmer)
6. Main preprocessing function
7. Dataset cleaning function
8. Load data & run cleaning
9. Slice to required columns
10. Shuffle
11. Label encoding
12. Train / Validation / Test split  ← **notebook ends here**

> **Output column:** `clean_text_stemmed`  
> **Saved files:** `train.csv`, `val.csv`, `test.csv`

---
## 1 · Install & Import Dependencies

In [15]:
!pip install emoji deep-translator nltk scikit-learn pandas numpy joblib

In [16]:
import os
import re
import sys
import warnings
import unicodedata
import pickle
import time
from collections import Counter

warnings.filterwarnings("ignore")
os.makedirs("outputs", exist_ok=True)

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack, csr_matrix, save_npz

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

import nltk
nltk.download("stopwords", quiet=True)
from nltk.stem.isri import ISRIStemmer
from nltk.corpus import stopwords

# Optional: emoji library
try:
    import emoji as emoji_lib
    EMOJI_OK = True
except ImportError:
    EMOJI_OK = False
    print("[WARN] `emoji` not installed. Run: pip install emoji")

# Optional: deep_translator for English→Arabic translation
try:
    from deep_translator import GoogleTranslator
    TRANSLATOR_OK = True
except ImportError:
    TRANSLATOR_OK = False
    print("[WARN] `deep_translator` not installed. English→Arabic translation disabled.")

print("[OK] All imports complete.")

[OK] All imports complete.


---
## 2 · Configuration Block

In [17]:
# ============================================================
# CONFIGURATION — edit these values before running
# ============================================================

# --- Paths ---
Dataset_path   = "/content/Multi-dialect Arabic dataset.xlsx"
SHEET_INDEX    = 17                       # First Excel sheet index
OUTPUT_DIR     = "outputs"                # directory for all saved files
OUTPUT_PREFIX  = "outputs/stemmed"        # prefix for output filenames

# --- Column names ---
TEXT_COL       = "comment"               # column containing raw text
LABEL_COL      = "label"                 # column containing class labels
META_COLS      = ["dialect", "platform"] # optional categorical meta columns (or [])

# --- Preprocessing toggles ---
NORMALIZE_TA_MARBUTA = True    # ة → ه
NORMALIZE_HAMZA      = True    # ؤ / ئ → ء
NORMALIZE_PERSIAN    = True    # چ→ج, گ→ك, پ→ب, ڤ→ف, ې→ي
REMOVE_EMOJIS        = True    # remove emoji characters
REMOVE_NON_ARABIC    = True    # strip non-Arabic / non-digit characters
TRANSLATE_ENGLISH    = False   # translate Latin words to Arabic (slow)
DROP_NEUTRAL_LABEL   = True    # drop rows with 'neutral' label

# --- Stemming ---
APPLY_STEMMING       = True    # must be True for this notebook

# --- Split ratios ---
TEST_SIZE      = 0.15
VAL_SIZE       = 0.15
RANDOM_STATE   = 42

print("[OK] Configuration loaded.")
print(f"     APPLY_STEMMING = {APPLY_STEMMING}")

[OK] Configuration loaded.
     APPLY_STEMMING = True


---
## 3 · Constants, Regex Patterns & Stopwords

In [18]:
# Arabic-Indic & Extended Arabic-Indic → Western numerals
_AR_INDIC     = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
_EXT_AR_INDIC = str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789")

# Arabizi digit→letter substitutions common in social media
ARABIZI_DIGIT_MAP = {
    "2": "ء", "3": "ع", "5": "خ", "6": "ط",
    "7": "ح", "8": "ق", "9": "ص",
}

# Persian letter normalization map
PERSIAN_NORM_MAP = str.maketrans({
    "چ": "ج",   # Persian che → Arabic jeem
    "گ": "ك",   # Persian gaf → Arabic kaf
    "پ": "ب",   # Persian pe  → Arabic ba
    "ڤ": "ف",   # Persian va  → Arabic fa
    "ې": "ي",   # Pashto ye   → Arabic ya
})

# Compiled regex patterns
RE_URL       = re.compile(r"https?://\S+|www\.\S+")
RE_MENTION   = re.compile(r"@\w+")
RE_HASHTAG   = re.compile(r"#(\w+)")           # keep hashtag word, drop #
RE_FLAG      = re.compile(r"[\U0001F1E6-\U0001F1FF]{2}")
RE_EMOJI_FB  = re.compile(r"[\U00010000-\U0010ffff]", flags=re.UNICODE)
RE_TASHKEEL  = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED]")
RE_TATWEEL   = re.compile(r"\u0640")
RE_OBFUSC_AR = re.compile(r"(?<=[\u0600-\u06FF])([^\u0600-\u06FF\w\s]+)(?=[\u0600-\u06FF])")
RE_ELONGATE  = re.compile(r"(.)\1{2,}")
RE_LATIN     = re.compile(r"^[A-Za-z]{2,}$")
RE_AR_TOKEN  = re.compile(r"[\u0600-\u06FF]+|\d+")
RE_REPUNCT   = re.compile(r"([!?.,،؛؟]){2,}")

# Ligature normalization
LIGATURE_MAP = str.maketrans({
    "ﷲ": "الله",
    "ﻷ": "لا",
    "ﻹ": "لا",
    "ﻵ": "لا",
    "ﻻ": "لا",
})

# NLTK Arabic stopwords + dialectal additions
try:
    AR_STOPWORDS = set(stopwords.words("arabic"))
except Exception:
    AR_STOPWORDS = set()

_DIALECT_STOPS = {
    "يعني", "هيك", "هكذا", "هكي", "كذا", "زي", "زيي", "ذي",
    "هذا", "هاذا", "هاد", "هاي", "واحد", "شي", "حاجة",
    "لما", "لمن", "لحد", "ومن", "وفي", "وعلى",
    "يا", "يالي", "اللي", "إللي", "لي", "الي",
    "فيه", "فيها", "فيهم", "منه", "منها", "منهم", "عليه", "عليها",
}
AR_STOPWORDS.update(_DIALECT_STOPS)

# ISRIStemmer instance
stemmer = ISRIStemmer()

print(f"[OK] Constants loaded. Arabic stopwords count: {len(AR_STOPWORDS)}")

[OK] Constants loaded. Arabic stopwords count: 722


---
## 4 · Helper Preprocessing Functions

In [19]:
def unicode_normalize(text: str) -> str:
    """Apply NFKC unicode normalization and collapse whitespace."""
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    return re.sub(r"\s+", " ", text).strip()


def convert_arabic_indic(text: str) -> str:
    """Convert Arabic-Indic and Extended Arabic-Indic digits to Western digits."""
    return text.translate(_AR_INDIC).translate(_EXT_AR_INDIC)


def normalize_ligatures(text: str) -> str:
    """Expand common Arabic ligatures: ﷲ→الله, ﻻ→لا etc."""
    return text.translate(LIGATURE_MAP)


def extract_flag_codes(text: str):
    """Remove country flag emoji and return (cleaned_text, list_of_country_codes)."""
    codes = []
    for f in RE_FLAG.findall(text):
        try:
            if len(f) == 2:
                codes.append("".join(chr(ord(c) - 0x1F1E6 + ord("A")) for c in f))
        except Exception:
            pass
    return RE_FLAG.sub(" ", text), codes


def extract_emojis(text: str):
    """Remove emoji characters; return (cleaned_text, emoji_count)."""
    if EMOJI_OK:
        count = sum(1 for c in text if c in emoji_lib.EMOJI_DATA)
        text  = emoji_lib.replace_emoji(text, replace=" ")
    else:
        matches = RE_EMOJI_FB.findall(text)
        count   = len(matches)
        text    = RE_EMOJI_FB.sub(" ", text)
    return text, count


def remove_urls_mentions(text: str) -> str:
    """Remove HTTP/HTTPS URLs and @mentions."""
    return RE_MENTION.sub(" ", RE_URL.sub(" ", text))


def clean_hashtags(text: str) -> str:
    """Strip the # symbol but keep the hashtag word."""
    return RE_HASHTAG.sub(r" \1 ", text)


def remove_tashkeel(text: str) -> str:
    """Remove Arabic diacritics (tashkeel) and tatweel elongation char."""
    return RE_TATWEEL.sub("", RE_TASHKEEL.sub("", text))


def normalize_arabic_letters(text: str,
                              ta_marbuta: bool = True,
                              hamza: bool = True,
                              persian: bool = True) -> str:
    """Normalize Arabic letter variants.

    - Alef variants (إأٱآ) → ا
    - Alef Maqsura (ى) → ي
    - Ta Marbuta (ة) → ه  [configurable]
    - Hamza variants (ؤئ) → ء  [configurable]
    - Persian letters (چگپڤې) → Arabic equivalents  [configurable]
    """
    text = re.sub(r"[إأٱآ]", "ا", text)      # Alef variants
    text = re.sub(r"ى", "ي", text)             # Alef Maqsura
    if ta_marbuta:
        text = re.sub(r"ة", "ه", text)         # Ta Marbuta
    if hamza:
        text = re.sub(r"[ؤئ]", "ء", text)     # Hamza forms
    if persian:
        text = text.translate(PERSIAN_NORM_MAP) # Persian letter normalization
    return text


def deobfuscate(text: str) -> str:
    """Remove punctuation/separators injected between Arabic letters (obfuscation)."""
    prev = None
    while prev != text:
        prev = text
        text = RE_OBFUSC_AR.sub("", text)
    return text


def fix_fragmented_arabic(text: str) -> str:
    """Merge runs of isolated single Arabic characters (e.g. 'ت ا غ ي ر' → 'تاغير')."""
    tokens = text.split(" ")
    result, i = [], 0
    while i < len(tokens):
        tok = tokens[i]
        if len(tok) == 1 and re.match(r"[\u0600-\u06FF]", tok):
            run = [tok]
            j = i + 1
            while j < len(tokens) and len(tokens[j]) == 1 and re.match(r"[\u0600-\u06FF]", tokens[j]):
                run.append(tokens[j])
                j += 1
            result.append("".join(run) if len(run) >= 2 else tok)
            i = j if len(run) >= 2 else i + 1
        else:
            result.append(tok)
            i += 1
    return " ".join(result)


def collapse_elongation(text: str) -> str:
    """Reduce any character repeated 3+ times to at most 2 repetitions."""
    return RE_ELONGATE.sub(r"\1\1", text)


def handle_arabizi_tokens(text: str) -> str:
    """Convert Arabizi digit-letter tokens using common substitution map."""
    tokens, result = text.split(), []
    for tok in tokens:
        if tok.isdigit():
            result.append(tok)
        elif re.search(r"\d", tok) and re.search(r"[A-Za-z\u0600-\u06FF]", tok):
            result.append("".join(ARABIZI_DIGIT_MAP.get(ch, ch) for ch in tok))
        else:
            result.append(tok)
    return " ".join(result)


def remove_non_arabic_digits(text: str) -> str:
    """Keep only Arabic script and digits; strip everything else."""
    text = re.sub(r"[^\u0600-\u06FF0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


# ---- Batch English→Arabic translation helper ----

def batch_translate_english(text_series, translate: bool = True) -> dict:
    """Build a lookup dict {english_word: arabic_translation} from the corpus."""
    if not translate or not TRANSLATOR_OK:
        return {}
    print("[INFO] Extracting unique English words for batch translation …")
    all_english = {w.lower() for text in text_series
                   for w in str(text).split() if RE_LATIN.match(w)}
    lookup, batch_size = {}, 50
    word_list = list(all_english)
    for i in range(0, len(word_list), batch_size):
        batch = word_list[i: i + batch_size]
        try:
            joined     = " | ".join(batch)
            translated = GoogleTranslator(source="auto", target="ar").translate(joined)
            for w, t in zip(batch, translated.split("|")):
                lookup[w] = t.strip()
        except Exception:
            pass
    print(f"[INFO] Translated {len(lookup)} unique English terms.")
    return lookup


def apply_translation_lookup(text: str, lookup: dict) -> str:
    """Replace recognized Latin tokens with their Arabic translations."""
    tokens = text.split()
    return " ".join(lookup.get(tok.lower(), tok) if RE_LATIN.match(tok) else tok
                    for tok in tokens)


print("[OK] Helper functions defined.")

[OK] Helper functions defined.


---
## 5 · Stemming Function (ISRIStemmer)

In [20]:
def stem_and_remove_stopwords(text: str) -> str:
    """Tokenize Arabic text, remove stopwords, and apply ISRIStemmer.

    - Pure digit tokens are kept as-is.
    - Stopwords are dropped.
    - All remaining Arabic tokens are stemmed via ISRIStemmer.
    """
    tokens    = RE_AR_TOKEN.findall(text)
    processed = []
    for token in tokens:
        if token.isdigit():
            processed.append(token)          # keep numbers unchanged
        elif token in AR_STOPWORDS:
            continue                          # drop stopwords
        else:
            try:
                processed.append(stemmer.stem(token))
            except Exception:
                processed.append(token)
    return " ".join(processed)


print("[OK] Stemming function defined (ISRIStemmer).")
# Quick smoke-test
test_word = "المتعلمون"
print(f"  Stem of '{test_word}' → '{stemmer.stem(test_word)}'")

[OK] Stemming function defined (ISRIStemmer).
  Stem of 'المتعلمون' → 'تعلم'


---
## 6 · Main Preprocessing Function

In [21]:
def preprocess_text_row(text: str, translation_lookup: dict) -> str:
    """Full per-row preprocessing pipeline (WITH stemming).

    Order:
      unicode normalize → Arabic-Indic digits → ligatures → flags →
      emojis → URLs/mentions → hashtags → translation → deobfuscate →
      fix fragments → tashkeel → letter normalize (incl. Persian) →
      elongation → Arabizi → non-Arabic strip → stem+stopwords →
      punctuation collapse → final whitespace
    """
    text = unicode_normalize(text)
    text = convert_arabic_indic(text)
    text = normalize_ligatures(text)
    text, _flags = extract_flag_codes(text)
    if REMOVE_EMOJIS:
        text, _em = extract_emojis(text)
    text = remove_urls_mentions(text)
    text = clean_hashtags(text)
    text = apply_translation_lookup(text, translation_lookup)
    text = deobfuscate(text)
    text = fix_fragmented_arabic(text)
    text = remove_tashkeel(text)
    text = normalize_arabic_letters(
        text,
        ta_marbuta=NORMALIZE_TA_MARBUTA,
        hamza=NORMALIZE_HAMZA,
        persian=NORMALIZE_PERSIAN,
    )
    text = collapse_elongation(text)
    text = handle_arabizi_tokens(text)
    if REMOVE_NON_ARABIC:
        text = remove_non_arabic_digits(text)
    # --- STEMMING STEP ---
    text = stem_and_remove_stopwords(text)
    text = RE_REPUNCT.sub(r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


print("[OK] preprocess_text_row defined.")

[OK] preprocess_text_row defined.


---
## 7 · Dataset Cleaning Function

In [22]:
def clean_dataset(df: pd.DataFrame,
                  text_col: str = TEXT_COL,
                  label_col: str = LABEL_COL,
                  translate: bool = TRANSLATE_ENGLISH,
                  drop_neutral: bool = DROP_NEUTRAL_LABEL,
                  save_prefix: str = OUTPUT_PREFIX) -> pd.DataFrame:
    """Clean the full dataset and produce the `clean_text_stemmed` column."""

    df = df.copy()
    df[text_col] = df[text_col].fillna("").astype(str)

    print(f"\n[STEP 0] Dataset shape before cleaning: {df.shape}")
    print(f"         Label distribution:\n{df[label_col].value_counts()}\n")

    # Step 1: Drop duplicate rows
    before = len(df)
    df = df.drop_duplicates(subset=[text_col])
    print(f"[STEP 1] Dropped {before - len(df)} duplicate rows. Remaining: {len(df)}")

    # Step 2: Optionally drop 'neutral' label rows
    if drop_neutral and "neutral" in df[label_col].str.lower().unique():
        df = df[df[label_col].str.lower() != "neutral"].reset_index(drop=True)
        print(f"[STEP 2] Dropped neutral labels. Remaining: {len(df)}")

    # Step 3: Drop rows with empty text
    df = df[df[text_col].str.strip() != ""].reset_index(drop=True)

    # Step 4: Build translation lookup once (efficiency)
    translation_lookup = batch_translate_english(df[text_col], translate=translate)

    # Step 5: Per-row cleaning
    print("[STEP 5] Cleaning and stemming text (this may take a moment) …")
    flag_counts, emoji_counts, hashtag_flags, cleaned_texts = [], [], [], []

    for text in df[text_col]:
        # --- collect metadata BEFORE full strip ---
        norm = unicode_normalize(text)
        norm = convert_arabic_indic(norm)
        norm = normalize_ligatures(norm)
        norm, flags = extract_flag_codes(norm)
        norm, em_cnt = extract_emojis(norm)
        hashtag_flags.append(int("#" in unicode_normalize(text)))
        flag_counts.append(len(flags))
        emoji_counts.append(em_cnt)
        # --- full preprocessing including stemming ---
        cleaned_texts.append(preprocess_text_row(text, translation_lookup))

    df["clean_text_stemmed"] = cleaned_texts
    df["flag_count"]         = flag_counts
    df["emoji_count"]        = emoji_counts
    df["has_emoji"]          = (df["emoji_count"] > 0).astype(int)
    df["has_hashtag"]        = hashtag_flags
    df["text_length"]        = df["clean_text_stemmed"].apply(lambda x: len(x.split()))

    # Step 6: Remove rows where cleaning produced empty text
    empty_mask = df["clean_text_stemmed"].str.strip() == ""
    print(f"[STEP 6] Removed {empty_mask.sum()} rows with empty text after cleaning.")
    df = df[~empty_mask].reset_index(drop=True)

    print(f"\n[INFO] Final dataset shape: {df.shape}")
    print(f"[INFO] Label distribution after preprocessing:\n{df[label_col].value_counts()}\n")

    # Log optional meta column distributions
    for mc in ["dialect", "platform"]:
        if mc in df.columns:
            print(f"[INFO] {mc.capitalize()} Distribution After Cleaning:\n{df[mc].value_counts()}\n")

    # Preview first 5 examples
    print("[INFO] Examples of cleaned+stemmed comments:")
    for i, (orig, clean) in enumerate(
            zip(df[text_col].head(5), df["clean_text_stemmed"].head(5)), 1):
        print(f"\n--- Example {i} ---")
        print(f"  Original : {orig}")
        print(f"  Cleaned  : {clean}")



    return df


print("[OK] clean_dataset function defined.")

[OK] clean_dataset function defined.


---
## 8 · Load Data & Run Cleaning

In [23]:
# Load dataset from Excel
df_raw = pd.read_excel(Dataset_path, sheet_name=SHEET_INDEX)
print(f"[OK] Loaded dataset: {df_raw.shape}")
print(df_raw)

[OK] Loaded dataset: (7627, 5)
                                                comment dialect platform  \
0     ماشاء الله يا حظج ويا هناج والله سلمى عليهم وق...  Kuwait        X   
1     بناتنا الجميلات 😍😍 حبيتهم وايد والشغل معاهم مت...  Kuwait        X   
2        حبيبتى الله يونسج بالعافيه وحمدالله على سلامتج  Kuwait        X   
3     حبيبتي فنانتنا الحلوة خفيفة الدم بنيتنا لولوة ...  Kuwait       IG   
4                تبارك الرحمن جميله ❤️ تكفين لا تتغيرين  Kuwait       IG   
...                                                 ...     ...      ...   
7622                هدشي مكتعتبروهش تفاهة وقلة الاداب\n   Libya       FB   
7623                           عيشة راجل مافيهش انوثة🤣🤣   Libya       FB   
7624  في نظرك انتي بس اما نحنا تعتبر ظاغية زي ابوها ...   Libya       FB   
7625                 وأنتي شن شكلك ياسمحة ..شكلك غوريلا   Libya       FB   
7626   وكان بلوفر ولا وحدة من الساقطات رااااك قلتي و...   Libya       FB   

      category     label  
0     Feminist  positive  
1 

In [24]:
# Run full dataset cleaning (with stemming)
df_clean = clean_dataset(df_raw)


[STEP 0] Dataset shape before cleaning: (7627, 5)
         Label distribution:
label
positive    3627
negative    3625
Neutral      371
Name: count, dtype: int64

[STEP 1] Dropped 19 duplicate rows. Remaining: 7608
[STEP 2] Dropped neutral labels. Remaining: 7239
[STEP 5] Cleaning and stemming text (this may take a moment) …
[STEP 6] Removed 0 rows with empty text after cleaning.

[INFO] Final dataset shape: (7238, 11)
[INFO] Label distribution after preprocessing:
label
positive    3622
negative    3616
Name: count, dtype: int64

[INFO] Dialect Distribution After Cleaning:
dialect
Sudanese     529
Yemen        528
Libya        528
Jordan       528
Saudi        528
Algeria      528
Iraq         527
Egypt        527
Palestine    526
Moroccan     526
Lebanon      525
Syria        525
Tunisia      524
Kuwait       132
Emirates     132
Bahrain       50
Qaṭar         40
Oman          35
Name: count, dtype: int64

[INFO] Platform Distribution After Cleaning:
platform
FB        3486
X       

---
## 9 · Slice to Required Columns

In [25]:
# Keep only the text and label columns for downstream use
df_clean = df_clean[["label", "clean_text_stemmed", "emoji_count", "has_emoji"]]

print(f"[OK] Sliced to columns: {list(df_clean.columns)}")
print(f"     Shape: {df_clean.shape}")
df_clean

[OK] Sliced to columns: ['label', 'clean_text_stemmed', 'emoji_count', 'has_emoji']
     Shape: (7238, 4)


,label,clean_text_stemmed,emoji_count,has_emoji
0,positive,اشء الل حظج ويا هنج ولل سلم علي وقل شكر قلب,1,1
1,positive,بنت جمل حبت ويد شغل معا تعه,3,1
2,positive,حبب الل نسج عفي وحمدالله علي متج,0,0
3,positive,حبب فنن حله خفف لدم بنت ولو لكه كوميد رقي يجب ...,0,0
4,positive,برك رحم جمل تكف تغر,1,1
...,...,...,...,...
7233,negative,هدش مكتعتبروهش فهه وقل ادب,0,0
7234,negative,عيش رجل مافيهش وثه,2,1
7235,negative,نظر انت اما نحن عبر ظغي ابو فعل نظر وجه فقر سه...,0,0
7236,negative,ونت شن شكل سمح شكل غوريل,0,0


---
## 10 · Shuffle

Shuffle the cleaned dataset with a fixed random seed before splitting.  
This removes any ordering bias that may exist in the raw data (e.g., all positives first).

In [26]:
RANDOM_SEED = 42

df_clean = df_clean.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"[SHUFFLE] Dataset shuffled with random_state={RANDOM_SEED}.")
print(f"          Shape: {df_clean.shape}")
print(f"          Label distribution after shuffle:\n{df_clean['label'].value_counts()}\n")

[SHUFFLE] Dataset shuffled with random_state=42.
          Shape: (7238, 4)
          Label distribution after shuffle:
label
positive    3622
negative    3616
Name: count, dtype: int64



---
## 11 · Label Encoding

Convert string labels (e.g., `positive`, `negative`) to integer IDs.  
Both the original string label and the integer `labels` column are kept in the dataset.  
The mapping is saved to `outputs/label_map.json` for use during training and inference.

In [27]:
import json

le = LabelEncoder()
df_clean["labels"] = le.fit_transform(df_clean["label"])

class_names = le.classes_.tolist()           # e.g., ['negative', 'positive']
label2id    = {c: int(i) for i, c in enumerate(class_names)}
id2label    = {int(i): c for i, c in enumerate(class_names)}

print(f"[LABEL] Classes  : {class_names}")
print(f"[LABEL] label2id : {label2id}")
print(f"[LABEL] id2label : {id2label}")
print()
print(df_clean[["label", "labels"]].value_counts().sort_index().to_string())

# Save the label map
label_map_path = f"{OUTPUT_DIR}/label_map.json"
with open(label_map_path, "w", encoding="utf-8") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, ensure_ascii=False, indent=2)
print(f"\n[SAVED] Label map → {label_map_path}")

# Drop the string label column — keep only integer labels
df_clean = df_clean.drop(columns="label")
LABEL_COL = "labels"

print(f"\n[OK] df_clean columns: {list(df_clean.columns)}")
df_clean

[LABEL] Classes  : ['negative', 'positive']
[LABEL] label2id : {'negative': 0, 'positive': 1}
[LABEL] id2label : {0: 'negative', 1: 'positive'}

label     labels
negative  0         3616
positive  1         3622

[SAVED] Label map → outputs/label_map.json

[OK] df_clean columns: ['clean_text_stemmed', 'emoji_count', 'has_emoji', 'labels']


,clean_text_stemmed,emoji_count,has_emoji,labels
0,تهل فخم حرم قدر,0,0,1
1,قرح بلد نهر شنو نتو وين مو عرق عقل هيج ضمر اكو,0,0,0
2,تعد ربح كبر ناء ملك دعر خمس شاء ركز دعر قنن,0,0,0
3,انت ضعف واه حقر انك تشه شهد قءد سمر رسك سيد رء...,0,0,1
4,سمع كذب حلفو اول حكم فسد كمل,0,0,0
...,...,...,...,...
7233,ولي عهد سعد رجل تكبر سمح بشش ولل شهد رجل حضرم ...,0,0,1
7234,ولل شكل الم ارد صعب الل يعن بلد بتج شعب بجن ال...,0,0,1
7235,جلس مثل الا نفس مثل,0,0,1
7236,ضلو حمم اتو وهي شكل هربو او ذرت,0,0,0


---
## 12 · Train / Validation / Test Split

Stratified split so each subset preserves the original class distribution.  
Each split is saved as a separate CSV file.

In [28]:
# ── Stratified split ──────────────────────────────────────────────────────────
y = df_clean[LABEL_COL].values

# 1st split: carve out the test set
df_temp, df_test, y_temp, y_test = train_test_split(
    df_clean, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

# 2nd split: carve out the validation set from the remaining data
relative_val = VAL_SIZE / (1.0 - TEST_SIZE)
df_train, df_val, y_train, y_val = train_test_split(
    df_temp, y_temp,
    test_size=relative_val,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

# Reset indices so each CSV has clean 0-based row numbers
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)


df_clean = df_clean[["labels", "clean_text_stemmed", "emoji_count", "has_emoji"]]
df_train = df_train[["labels", "clean_text_stemmed", "emoji_count", "has_emoji"]]
df_val = df_val[["labels", "clean_text_stemmed", "emoji_count", "has_emoji"]]
df_test = df_test[["labels", "clean_text_stemmed", "emoji_count", "has_emoji"]]

# ── Print split statistics ────────────────────────────────────────────────────
print(f"[SPLIT] Train : {len(df_train):>5} rows")
print(f"[SPLIT] Val   : {len(df_val):>5} rows")
print(f"[SPLIT] Test  : {len(df_test):>5} rows")
print(f"[SPLIT] Total : {len(df_train) + len(df_val) + len(df_test):>5} rows")

print("\n[INFO] Label distribution per split:")
for split_name, df_split in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    counts = df_split[LABEL_COL].value_counts().sort_index()
    dist   = {id2label[k]: v for k, v in counts.items()}
    print(f"  {split_name:5s}: {dist}")

# ── Save each split as CSV ────────────────────────────────────────────────────
df_clean_path = f"{OUTPUT_PREFIX}_data.csv"
train_path = f"{OUTPUT_PREFIX}_train.csv"
val_path   = f"{OUTPUT_PREFIX}_val.csv"
test_path  = f"{OUTPUT_PREFIX}_test.csv"


df_clean.to_csv( df_clean_path, index=False, encoding="utf-8-sig")
df_train.to_csv( train_path, index=False, encoding="utf-8-sig")
df_val.to_csv(  val_path,   index=False, encoding="utf-8-sig")
df_test.to_csv( test_path,  index=False, encoding="utf-8-sig")


print(f"\n[SAVED] df_clean set   ({len(df_clean)} rows) → {df_clean_path}")
print(f"\n[SAVED] Training set   ({len(df_train)} rows) → {train_path}")
print(f"[SAVED] Validation set ({len(df_val)} rows) → {val_path}")
print(f"[SAVED] Test set       ({len(df_test)} rows) → {test_path}")

# ── Quick preview ─────────────────────────────────────────────────────────────
print("\n[PREVIEW] First 3 rows of each split:")
print("\n--- Train ---")
display(df_train.head(3))
print("\n--- Val ---")
display(df_val.head(3))
print("\n--- Test ---")
display(df_test.head(3))

[SPLIT] Train :  5066 rows
[SPLIT] Val   :  1086 rows
[SPLIT] Test  :  1086 rows
[SPLIT] Total :  7238 rows

[INFO] Label distribution per split:
  Train: {'negative': 2531, 'positive': 2535}
  Val  : {'negative': 542, 'positive': 544}
  Test : {'negative': 543, 'positive': 543}

[SAVED] df_clean set   (7238 rows) → outputs/stemmed_data.csv

[SAVED] Training set   (5066 rows) → outputs/stemmed_train.csv
[SAVED] Validation set (1086 rows) → outputs/stemmed_val.csv
[SAVED] Test set       (1086 rows) → outputs/stemmed_test.csv

[PREVIEW] First 3 rows of each split:

--- Train ---


,labels,clean_text_stemmed,emoji_count,has_emoji
0,0,يوم اخر تكد بان عملاء عمل صلح صلح دول فهي خين ...,0,0
1,0,جمع سان دي مش طبع عمل خبط ناس كله فرض حكم تشف ...,0,0
2,0,تقد لحس خزي,0,0



--- Val ---


,labels,clean_text_stemmed,emoji_count,has_emoji
0,0,هده قسم حرب فضل ابم حكم قبر لهم جعل درك سفل نا...,0,0
1,1,عاش ملك عاش ملك عاش ملك ملي مره الل يخل لين ير...,3,1
2,0,الل دخلو واح كتبو عيطو رهم خطر,0,0



--- Test ---


,labels,clean_text_stemmed,emoji_count,has_emoji
0,1,فخر فيك كتر انو انت مثل لفء شبب علم تهل خير ول...,5,1
1,1,رغم كرث هبط علي ونس ودي ربي معاها،ولا عاش ونس خان,0,0
2,0,شء قزز وشذ يجب جرم لحم طفل جرءم,0,0
